In [ ]:
import os
import rawpy
from PIL import Image
import piexif
from tqdm import tqdm
import numpy as np

def extract_gps_from_exif(exif_dict):
    """
    Extract GPS IFD and Image DateTime tag from EXIF dict.
    Returns a minimal EXIF dict with these tags only.
    """
    gps_ifd = exif_dict.get("GPS", {})

    zeroth_ifd = {}
    # Tag 0x0132 is DateTime in 0th IFD
    datetime_tag = 0x0132
    if datetime_tag in exif_dict.get("0th", {}):
        zeroth_ifd[datetime_tag] = exif_dict["0th"][datetime_tag]

    return {
        "0th": zeroth_ifd,
        "GPS": gps_ifd
    }
input_dir = "D:\\Rothaut_Thesis\\Mai\\raw"
output_dir = "D:\\Rothaut_Thesis\\Mai\\jpg"
os.makedirs(output_dir, exist_ok=True)

for filename in tqdm(sorted(os.listdir(input_dir))):
    if not filename.lower().endswith(".dng"):
        continue
    
    filename = filename.replace("._","")
    out_name = filename.replace(".dng", ".jpg").replace(".DNG", ".jpg")
    jpgs = os.listdir(output_dir)
    if out_name in jpgs:
        continue
    dng_path = input_dir +"/"+filename
    with rawpy.imread(dng_path) as raw:
        rgb = raw.postprocess(exp_shift=2, no_auto_bright=True)
    img = Image.fromarray(rgb)
    exif_dict = piexif.load(dng_path)
    gps_only_exif = extract_gps_from_exif(exif_dict)
    exif_bytes = piexif.dump(gps_only_exif)
    out_path = os.path.join(output_dir, out_name)
    img.save(out_path, "jpeg", exif=exif_bytes, quality=100)

print("GPS-only EXIF transfer complete.")

100%|██████████| 495/495 [09:58<00:00,  1.21s/it]

GPS-only EXIF transfer complete.


In [51]:
from PIL import Image
import numpy as np
import os
import json
import re
import shutil
def rename_json_files(folder_path, numb):
    for filename in os.listdir(folder_path):
        if not filename.endswith(".png"):
            filepath = os.path.join(folder_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                json_id = data.get("task")['id']
                image_name = data.get("task")['data']['image'].split('DJI_')[1].split('.')[0]
                if json_id == int(numb):
                    num_list.append(image_name)
                    return str(image_name)

def convert2to1(img1, img2, indir, outdir):
    if img2 == None:
        numb = img1.split("-")[1]
        image1 = Image.open(os.path.join(indir, img1)).convert("L")
        image1_array = np.array(image1)
        image1_array[image1_array != 255] = 0
        merged_im = Image.fromarray(image1_array, mode="L")
        print("Checking for Filename of Task ", numb)
        numb = rename_json_files(indir, numb)
        print("New Number: ", numb)
        merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))
        return False
    if img1 == None:
        numb = img2.split("-")[1]
        image2 = Image.open(os.path.join(indir, img1)).convert("L")
        image2_array = np.array(image1)
        image2_array[image1_array != 255] = 0
        image2_array[image2_array == 255] = newg
        merged_im = Image.fromarray(image2_array, mode="L")
        print("Checking for Filename of Task ", numb)
        numb = rename_json_files(indir, numb)
        print("New Number: ", numb)
        merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))
        return False
    numb = img1.split("-")[1]
    image1 = Image.open(os.path.join(indir, img1)).convert("L")
    image2 = Image.open(os.path.join(indir, img2)).convert("L")
    image1_array = np.array(image1)
    image2_array = np.array(image2)
    newg = 150
    image2_array[image2_array != 255] = 0
    image1_array[image1_array != 255] = 0
    image2_array[image2_array == 255] = newg
    merged = np.maximum(image1_array, image2_array)
    merged_im = Image.fromarray(merged, mode="L")
    print("Checking for Filename of Task ", numb)
    numb = rename_json_files(indir, numb)
    print("New Number: ", numb)
    merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))
def searchimg(imglist, task):
    for f in imglist:
        if task in f:
            return f
indir ="D:\Rothaut_Thesis\Mai\masks"
outdir="C:/Users/dmz-user/Desktop/unet_wein/thesis/images/masks"
img1list = []
img2list = []
tasks = set()
num_list = []
for filename in sorted(os.listdir(indir)):
    if filename.endswith(".png"):
        numb = filename.split("-")[1]
        tasks.add(numb)
        if "-mit" in filename:
            img1list.append(filename)
        else:
            img2list.append(filename)


for i in tasks:
    img1 = searchimg(img1list, i)
    print(img1)
    img2 = searchimg(img2list, i)
    print(img2)
    convert2to1(img1,img2, indir, outdir)
for f in os.listdir("D:\Rothaut_Thesis\Mai\jpg"):
    for num in sorted(num_list):
        if num in f:
            shutil.copy(os.path.join("D:\Rothaut_Thesis\Mai\jpg", f),"C:/Users/dmz-user/Desktop/unet_wein/thesis/images/imgs/"+f)


task-1485-annotation-14-by-1-tag-mit-0.png
task-1485-annotation-14-by-1-tag-ohne-0.png
Checking for Filename of Task  1485
New Number:  0599
task-1354-annotation-30-by-1-tag-mit-0.png
None
Checking for Filename of Task  1354
New Number:  0468
task-1481-annotation-1-by-1-tag-mit-0.png
task-1481-annotation-1-by-1-tag-ohne-0.png
Checking for Filename of Task  1481
New Number:  0595
task-1355-annotation-31-by-1-tag-mit-0.png
None
Checking for Filename of Task  1355
New Number:  0469
task-1477-annotation-19-by-1-tag-mit-0.png
None
Checking for Filename of Task  1477
New Number:  0591
task-1258-annotation-22-by-1-tag-mit-0.png
None
Checking for Filename of Task  1258
New Number:  0366
task-1480-annotation-18-by-1-tag-mit-0.png
task-1480-annotation-18-by-1-tag-ohne-0.png
Checking for Filename of Task  1480
New Number:  0594
task-1465-annotation-5-by-1-tag-mit-0.png
task-1465-annotation-5-by-1-tag-ohne-0.png
Checking for Filename of Task  1465
New Number:  0579
task-1475-annotation-21-by-1-tag

In [ ]:
import os
from PIL import Image

def get_task_and_tag(filename):
    # Extrahiere Task und Tag aus dem Dateinamen
    base_name = os.path.basename(filename)
    task, tag = base_name.split('-')[1],base_name.split('-')[7]
    number = base_name.split('-')[8].split(".")[0]
    print(task, tag,number)
    return task, tag, int(number)  # Gib den Task, Tag und die Zahl zurück

def merge_images(image1, image2):
    # Vereine zwei Bilder (z.B. nebeneinander)
    new_width = image1.width + image2.width
    new_height = max(image1.height, image2.height)
    new_image = Image.new('RGB', (new_width, new_height))
    
    # Füge beide Bilder nebeneinander hinzu
    new_image.paste(image1, (0, 0))
    new_image.paste(image2, (image1.width, 0))
    
    return new_image

def process_images(image_folder):
    # Erstelle ein Dictionary, um Bilder nach Task und Tag zu gruppieren
    image_dict = {}
    
    for filename in os.listdir(image_folder):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            task, tag, number = get_task_and_tag(filename)
            key = (task, tag)
            
            # Speichern der Bilder nach Task und Tag
            if key not in image_dict:
                image_dict[key] = []
            image_dict[key].append((number, filename))
    
    # Nun die Bilder mit der niedrigeren Nummer benennen und ggf. vereinen
    for (task, tag), images in image_dict.items():
        # Sortiere nach der Nummer
        images.sort(key=lambda x: x[0])
        
        # Das Bild mit der niedrigsten Nummer benennen
        first_image_filename = images[0][1]
        first_image = Image.open(os.path.join(image_folder, first_image_filename))
        
        if len(images) == 2:
            second_image_filename = images[1][1]
            second_image = Image.open(os.path.join(image_folder, second_image_filename))
            
            # Vereine die beiden Bilder
            merged_image = merge_images(first_image, second_image)
            merged_image.save(os.path.join(image_folder, f"{task}-tag-{tag}.png"))
        else:
            # Falls nur ein Bild vorhanden ist, belasse es
            first_image.save(os.path.join(image_folder, f"{task}-tag-{tag}.png"))

# Beispiel: den Pfad zum Ordner angeben
image_folder = "D:\Rothaut_Thesis\Mai\masks"
process_images(image_folder)

1459 mit 0
1464 mit 0
1464 ohne 0
1465 mit 0
1465 ohne 0
1466 mit 0
1466 ohne 0
1478 mit 0
1478 ohne 0
1479 mit 0
1479 mit 1
1479 mit 2
1479 ohne 0
1481 mit 0
1481 mit 1
1481 ohne 0


In [35]:
import json
import numpy as np
import pandas as pd

# Load the uploaded file
file_path = 'data/masks/6'  # Adjust the file path accordingly
with open(file_path, 'r') as file:
    data = json.load(file)

# Initialize image size and label to grayscale mapping
image_size = (8064, 6048)  # Image dimensions (8064x6048)
label_to_gray = {'ohne': 255}  # 'ohne' label is represented by 255 in grayscale

# Extract annotations from the 'result' field
annotations = data['result']

# Initialize the NumPy array with zeros (black background)
image_array = np.zeros(image_size, dtype=np.uint8)

# Process each annotation
for annotation in annotations:
    label = annotation.get('brushlabels', [None])[0]
    if label in label_to_gray:
        # Get the grayscale value for the label
        gray_value = label_to_gray[label]
        
        # Extract the points (coordinates) of the annotation
        points = annotation.get('points', [])
        
        # If there are points, mark them in the grayscale array
        for point in points:
            x, y = int(point[0]), int(point[1])
            if 0 <= x < image_size[0] and 0 <= y < image_size[1]:
                image_array[y, x] = gray_value  # Y, X because the typical image format is row-major

# Convert the NumPy array to a DataFrame for easier viewing
image_df = pd.DataFrame(image_array)
print(np.unique(image_array))
# Display the resulting DataFrame (this can be customized if you need to save it or visualize in another way)

[0]
